In [14]:
import pandas as pd
import numpy as np
import csv

In [15]:
file_path = r'/Users/maciek/Documents/Projekty/netflix-datawarehouse-ML/databases/bronze/watch_history.csv'

df_bronze = pd.read_csv(file_path, sep = ',')

df_silver = df_bronze.copy()

In [27]:
df_silver.head()

,session_id,user_id,movie_id,watch_date,device_type,watch_duration_minutes,progress_percentage,action,quality,location_country,is_download,user_rating,is_watch_duration_minutes_missing,is_progress_percentage_null
0,session_000001,user_07271,movie_0511,2025-11-13,Tablet,63.9,34.6,completed,HD,USA,False,NaN,0,0
1,session_000002,user_00861,movie_0588,2025-02-26,Laptop,120.1,44.2,started,HD,USA,False,NaN,0,0
2,session_000003,user_05391,movie_0694,2024-12-15,Desktop,572.1,84.7,started,HD,Canada,False,1.0,0,0
3,session_000004,user_05192,movie_0234,2024-09-30,Desktop,395.3,89.9,completed,SD,USA,False,5.0,0,0
4,session_000005,user_05735,movie_0390,2024-08-04,Tablet,14.6,6.2,completed,HD,USA,False,NaN,0,0


In [16]:
df_silver.isnull().sum()

session_id                    0
user_id                       0
movie_id                      0
watch_date                    0
device_type                   0
watch_duration_minutes    12332
progress_percentage        8514
action                        0
quality                       0
location_country              0
is_download                   0
user_rating               83903
dtype: int64

In [17]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   session_id              105000 non-null  object 
 1   user_id                 105000 non-null  object 
 2   movie_id                105000 non-null  object 
 3   watch_date              105000 non-null  object 
 4   device_type             105000 non-null  object 
 5   watch_duration_minutes  92668 non-null   float64
 6   progress_percentage     96486 non-null   float64
 7   action                  105000 non-null  object 
 8   quality                 105000 non-null  object 
 9   location_country        105000 non-null  object 
 10  is_download             105000 non-null  bool   
 11  user_rating             21097 non-null   float64
dtypes: bool(1), float64(3), object(8)
memory usage: 8.9+ MB


In [18]:
df_silver['device_type'].value_counts()

device_type
Desktop     21125
Tablet      21098
Smart TV    21002
Mobile      20947
Laptop      20828
Name: count, dtype: int64

In [21]:
median_duration_minutes = df_silver['watch_duration_minutes'].median()

df_silver['is_watch_duration_minutes_missing'] = df_silver['watch_duration_minutes'].isnull().astype(int)

#Verification
df_silver['is_watch_duration_minutes_missing'].value_counts()

is_watch_duration_minutes_missing
0    92668
1    12332
Name: count, dtype: int64

In [22]:
df_silver['watch_duration_minutes'] = df_silver['watch_duration_minutes'].fillna(median_duration_minutes)

df_silver['watch_duration_minutes'].isnull().sum()

np.int64(0)

In [25]:
df_silver['is_progress_percentage_null'] = df_silver['progress_percentage'].isnull().astype(int)

df_silver['is_progress_percentage_null'].value_counts()

is_progress_percentage_null
0    96486
1     8514
Name: count, dtype: int64

In [ ]:
#Fill with 0, later creates join between movies and watchhistory to create percentage_progress 
#that will replace it with watch_duration_minutes/total_minutes_of_films

df_silver['progress_percentage'] = df_silver['progress_percentage'].fillna(0)

df_silver['progress_percentage'].isnull().sum()

np.int64(0)

In [30]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 15 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   session_id                         105000 non-null  object 
 1   user_id                            105000 non-null  object 
 2   movie_id                           105000 non-null  object 
 3   watch_date                         105000 non-null  object 
 4   device_type                        105000 non-null  object 
 5   watch_duration_minutes             105000 non-null  float64
 6   progress_percentage                105000 non-null  float64
 7   action                             105000 non-null  object 
 8   quality                            105000 non-null  object 
 9   location_country                   105000 non-null  object 
 10  is_download                        105000 non-null  bool   
 11  user_rating                        1050

In [28]:
df_silver['is_user_rating_missing'] = df_silver['user_rating'].isnull().astype(int)

df_silver['user_rating'] = df_silver['user_rating'].fillna(0)

In [29]:
df_silver['watch_date'] = pd.to_datetime(df_silver['watch_date'], errors = 'coerce')

df_silver['watch_date'] = (df_silver['watch_date'].dt.strftime('%Y-%m-%d'))

In [31]:
df_silver['is_download'] = df_silver['is_download'].astype(int)

In [35]:
output_file = r'/Users/maciek/Documents/Projekty/netflix-datawarehouse-ML/databases/silver/netflix_silver_layer_watch_history.csv'

df_silver.to_csv(
    output_file,
    sep = ',',
    decimal = '.',
    index = False,
    encoding = 'utf-8',
    lineterminator = '\n',
    quoting = csv.QUOTE_MINIMAL
)

In [33]:
import os
print(os.getcwd())

/Users/maciek/Documents/Projekty/netflix-datawarehouse-ML/notebooks
